In [ ]:
Question 1: The Biased Credit Gate
Scenario: A Hyderabad NBFC approved only about 35% of its past loan applicants, and it has repayment outcomes only for those approved.
Now it wants to expand lending to applicants the old policy would have rejected. A wrong decision on a defaulter costs 8x more than losing
a good customer. Leadership asks how you will judge your system’s quality before it makes a single live decision on the previously rejected group.

In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("credit_applications.csv")
df

,app_id,application_date,age,city,employment_type,employment_years,monthly_income,existing_emi,credit_score,enquiries_6m,loan_amt,tenure_months,loan_purpose,past_decision,rejection_reason,outcome_12m
0,A115474,2023-11-01,35,Indore,Salaried,4.2,59200,0,702,2.0,380000,48,Wedding,Approved,NaN,Repaid
1,A119062,2023-11-01,38,Bengaluru,Self-employed,4.4,45500,0,668,1.0,290000,24,Debt consolidation,Rejected,Policy score below cut-off,Unknown
2,A107362,2023-11-01,34,Pune,Salaried,9.5,44200,0,743,4.0,150000,12,Wedding,Rejected,Policy score below cut-off,Unknown
3,A106905,2023-11-01,31,Vijayawada,Salaried,4.5,34400,0,684,1.0,270000,48,Debt consolidation,Approved,NaN,Repaid
4,A118676,2023-11-01,35,Pune,Gig/Contract,12.7,41700,0,647,5.0,240000,24,Travel,Rejected,Low credit score,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,A102261,2025-10-31,41,Chennai,Self-employed,5.2,52100,15500,621,2.0,290000,12,Debt consolidation,Rejected,Low credit score,Unknown
19996,A108146,2025-10-31,47,Chennai,Gig/Contract,3.4,14800,3800,508,2.0,110000,24,Debt consolidation,Rejected,Low credit score,Unknown
19997,A108702,2025-10-31,44,Vijayawada,Self-employed,10.4,55000,0,700,4.0,370000,24,Wedding,Rejected,Policy score below cut-off,Unknown
19998,A110842,2025-10-31,32,Hyderabad,Self-employed,1.0,35800,9500,633,5.0,250000,36,Business,Rejected,Low credit score,Unknown


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   app_id            20000 non-null  str    
 1   application_date  20000 non-null  str    
 2   age               20000 non-null  int64  
 3   city              20000 non-null  str    
 4   employment_type   20000 non-null  str    
 5   employment_years  19630 non-null  float64
 6   monthly_income    20000 non-null  int64  
 7   existing_emi      20000 non-null  int64  
 8   credit_score      20000 non-null  int64  
 9   enquiries_6m      19724 non-null  float64
 10  loan_amt          20000 non-null  int64  
 11  tenure_months     20000 non-null  int64  
 12  loan_purpose      20000 non-null  str    
 13  past_decision     20000 non-null  str    
 14  rejection_reason  13000 non-null  str    
 15  outcome_12m       20000 non-null  str    
dtypes: float64(2), int64(6), str(8)
memory usage: 3.8 M

In [6]:
df['enquiries_6m'].describe()

count    19724.000000
mean         2.233776
std          1.770012
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         12.000000
Name: enquiries_6m, dtype: float64

In [7]:
df.head()

,app_id,application_date,age,city,employment_type,employment_years,monthly_income,existing_emi,credit_score,enquiries_6m,loan_amt,tenure_months,loan_purpose,past_decision,rejection_reason,outcome_12m
0,A115474,2023-11-01,35,Indore,Salaried,4.2,59200,0,702,2.0,380000,48,Wedding,Approved,NaN,Repaid
1,A119062,2023-11-01,38,Bengaluru,Self-employed,4.4,45500,0,668,1.0,290000,24,Debt consolidation,Rejected,Policy score below cut-off,Unknown
2,A107362,2023-11-01,34,Pune,Salaried,9.5,44200,0,743,4.0,150000,12,Wedding,Rejected,Policy score below cut-off,Unknown
3,A106905,2023-11-01,31,Vijayawada,Salaried,4.5,34400,0,684,1.0,270000,48,Debt consolidation,Approved,NaN,Repaid
4,A118676,2023-11-01,35,Pune,Gig/Contract,12.7,41700,0,647,5.0,240000,24,Travel,Rejected,Low credit score,Unknown


In [8]:
df["application_date"] = pd.to_datetime(df["application_date"])

In [9]:
print(df["past_decision"].value_counts())
print(df["outcome_12m"].value_counts())

past_decision
Rejected    13000
Approved     7000
Name: count, dtype: int64
outcome_12m
Unknown      13000
Repaid        6378
Defaulted      622
Name: count, dtype: int64


In [10]:
print(df.groupby("past_decision")[["credit_score", 
                                   "monthly_income", 
                                   "existing_emi", 
                                   "loan_amt"]].mean().round(0))

               credit_score  monthly_income  existing_emi  loan_amt
past_decision                                                      
Approved              725.0         52243.0        2858.0  219231.0
Rejected              641.0         41552.0        7967.0  196242.0


In [11]:
# People we have answers for
labeled = df[df["past_decision"] == "Approved"].copy()

# People we don't have answers for
unlabeled = df[df["past_decision"] == "Rejected"].copy()

In [12]:
# Turn the outcome into numbers: 1 = Defaulted, 0 = Repaid
labeled["default"] = (labeled["outcome_12m"] == "Defaulted").astype(int)

print(len(labeled), len(unlabeled))
print(labeled["default"].mean().round(3))

7000 13000
0.089


In [13]:
cols = ["credit_score", "monthly_income", "existing_emi", "loan_amt"]

# Spread of values for approved people
print(labeled[cols].describe().round(0))

# Spread of values for rejected people
print(unlabeled[cols].describe().round(0))

# Lowest credit score our system has ever seen
min_score = labeled["credit_score"].min()
print("Lowest approved score:", min_score)

# What share of rejected people score below that?
print((unlabeled["credit_score"] < min_score).mean().round(3))

       credit_score  monthly_income  existing_emi   loan_amt
count        7000.0          7000.0        7000.0     7000.0
mean          725.0         52243.0        2858.0   219231.0
std            43.0         27809.0        5995.0   162478.0
min           640.0         12000.0           0.0    50000.0
25%           693.0         32700.0           0.0   100000.0
50%           720.0         46200.0           0.0   180000.0
75%           753.0         64900.0        3500.0   290000.0
max           900.0        381100.0       99300.0  1500000.0
       credit_score  monthly_income  existing_emi   loan_amt
count       13000.0         13000.0       13000.0    13000.0
mean          641.0         41552.0        7967.0   196242.0
std            51.0         22650.0       10098.0   144507.0
min           399.0         12000.0           0.0    50000.0
25%           610.0         25900.0           0.0    90000.0
50%           643.0         36600.0        5100.0   160000.0
75%           675.0     

In [14]:
labeled

,app_id,application_date,age,city,employment_type,employment_years,monthly_income,existing_emi,credit_score,enquiries_6m,loan_amt,tenure_months,loan_purpose,past_decision,rejection_reason,outcome_12m,default
0,A115474,2023-11-01,35,Indore,Salaried,4.2,59200,0,702,2.0,380000,48,Wedding,Approved,NaN,Repaid,0
3,A106905,2023-11-01,31,Vijayawada,Salaried,4.5,34400,0,684,1.0,270000,48,Debt consolidation,Approved,NaN,Repaid,0
18,A114017,2023-11-01,37,Vijayawada,Salaried,4.3,75500,3200,781,5.0,390000,60,Wedding,Approved,NaN,Repaid,0
23,A102315,2023-11-01,30,Warangal,Gig/Contract,3.6,49500,4800,723,0.0,380000,36,Medical,Approved,NaN,Defaulted,1
24,A104690,2023-11-02,35,Bengaluru,Self-employed,9.5,92200,18700,795,0.0,230000,60,Debt consolidation,Approved,NaN,Repaid,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19980,A103206,2025-10-31,44,Bengaluru,Salaried,12.8,82900,0,773,0.0,560000,24,Travel,Approved,NaN,Repaid,0
19982,A104421,2025-10-31,31,Nagpur,Salaried,4.1,29300,0,823,1.0,220000,24,Travel,Approved,NaN,Repaid,0
19990,A114983,2025-10-31,31,Bengaluru,Salaried,7.6,56000,0,665,0.0,250000,24,Business,Approved,NaN,Repaid,0
19991,A113189,2025-10-31,40,Hyderabad,Self-employed,3.3,61300,0,805,0.0,70000,48,Education,Approved,NaN,Repaid,0


In [15]:
y = labeled["default"]
approve_all_cost = 8 * (y == 1).sum()

# Reject everyone: every good customer is a mistake, costing 1 each
reject_all_cost = 1 * (y == 0).sum()

print("Approve all:", approve_all_cost)
print("Reject all:", reject_all_cost)


Approve all: 4976
Reject all: 6378


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

In [18]:
num_cols = ["age", "employment_years", "monthly_income", "existing_emi",
            "credit_score", "enquiries_6m", "loan_amt", "tenure_months"]
cat_cols = ["city", "employment_type", "loan_purpose"]

In [23]:
X = labeled[num_cols + cat_cols]
y = labeled["default"]

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [25]:
for col in num_cols:
    median = X_train[col].median()
    X_train[col] = X_train[col].fillna(median)
    X_test[col] = X_test[col].fillna(median)

# 2. Text to numbers
X_train = pd.get_dummies(X_train, columns=cat_cols)
X_test = pd.get_dummies(X_test, columns=cat_cols)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [26]:
# 3. Scale
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# 4. Train
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [28]:
p_default = model.predict_proba(X_test)[:, -1]

In [29]:
p_default

array([0.08992552, 0.05166138, 0.09943146, ..., 0.02758714, 0.05608056,
       0.02978037], shape=(1400,))

In [43]:
# 1 = approve, 0 = reject
approve = (p_default < 1/9).astype(int)

# Count the two kinds of mistakes
bad_approved = ((y_test == 1) & (approve == 1)).sum()    # defaulters we approved
good_rejected = ((y_test == 0) & (approve == 0)).sum()   # good people we rejected

model_cost = 8 * bad_approved + 1 * good_rejected

In [46]:
print(bad_approved)
print(good_rejected)

66
356


In [42]:

# The two silly strategies, on the same test people
approve_all_cost = 8 * (y_test == 1).sum()
reject_all_cost = 1 * (y_test == 0).sum()

print("Model cost:  ", model_cost)
print("Approve all: ", approve_all_cost)
print("Reject all:  ", reject_all_cost)
print("Share approved:", approve.mean().round(3))


Model cost:   884
Approve all:  992
Reject all:   1276
Share approved: 0.704


In [56]:
# Put the chances and real answers in one table
check = pd.DataFrame({"p_default": p_default, "defaulted": y_test.values})
check

,p_default,defaulted
0,0.089926,0
1,0.051661,1
2,0.099431,0
3,0.079093,0
4,0.091127,0
...,...,...
1395,0.023766,0
1396,0.085463,0
1397,0.027587,0
1398,0.056081,0


In [63]:

# Split into 5 equal groups, from safest to riskiest
check["group"] = pd.qcut(check["p_default"], 5, labels=[1, 2, 3, 4, 5])

# Average of each column, per group
print(check.groupby("group", observed=True).mean().round(3))

       p_default  defaulted
group                      
1          0.032      0.032
2          0.060      0.061
3          0.084      0.079
4          0.111      0.129
5          0.164      0.143


In [64]:
# Question 1: how many rejected people are outside what the model has seen?
lowest_approved_score = labeled["credit_score"].min()
outside = (unlabeled["credit_score"] < lowest_approved_score).mean()
print("Lowest approved credit score:", lowest_approved_score)
print("Share of rejected people below that:", round(outside, 3))

# Prepare the rejected people exactly like the training people
X_rej = unlabeled[num_cols + cat_cols].copy()
for col in num_cols:
    X_rej[col] = X_rej[col].fillna(labeled[col].median())
X_rej = pd.get_dummies(X_rej, columns=cat_cols)
X_rej = X_rej.reindex(columns=X_train.columns, fill_value=0)
X_rej[num_cols] = scaler.transform(X_rej[num_cols])

Lowest approved credit score: 640
Share of rejected people below that: 0.472


In [61]:

# what does the model say about them?
p_rej = model.predict_proba(X_rej)[:, 1]
print("Average chance of default, approved test people:", round(p_default.mean(), 3))
print("Average chance of default, rejected people:", round(p_rej.mean(), 3))
print("Share of rejected people the model would approve:", round((p_rej < 1/9).mean(), 3))

Average chance of default, approved test people: 0.09
Average chance of default, rejected people: 0.286
Share of rejected people the model would approve: 0.078


In [65]:
unlabeled["p_default"] = p_rej
unlabeled["model_approves"] = (p_rej < 1/9)

# Is the person inside the credit score range the model has seen?
unlabeled["seen_range"] = unlabeled["credit_score"] >= labeled["credit_score"].min()

print(pd.crosstab(unlabeled["seen_range"], unlabeled["model_approves"]))

model_approves  False  True 
seen_range                  
False            6115     15
True             5868   1002


In [66]:
# A cost function so we don't repeat ourselves
def total_cost(y_true, approve):
    y_true, approve = np.asarray(y_true), np.asarray(approve)
    return 8 * ((y_true == 1) & (approve == 1)).sum() + 1 * ((y_true == 0) & (approve == 0)).sum()

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Split approved people by credit score
cut = labeled["credit_score"].quantile(0.25)
core = labeled[labeled["credit_score"] >= cut]   # safer 75%: train on these
edge = labeled[labeled["credit_score"] < cut]    # riskiest 25%: test on these

# Same cleaning as before, learned from "core" only
med = core[num_cols].median()

def make_X(d):
    X = d[num_cols + cat_cols].copy()
    X[num_cols] = X[num_cols].fillna(med)
    return pd.get_dummies(X, columns=cat_cols)

Xc = make_X(core)
Xe = make_X(edge).reindex(columns=Xc.columns, fill_value=0)

sc = StandardScaler()
Xc[num_cols] = sc.fit_transform(Xc[num_cols])
Xe[num_cols] = sc.transform(Xe[num_cols])

# Train on core, predict on edge
edge_model = LogisticRegression(max_iter=1000).fit(Xc, core["default"])
p_edge = edge_model.predict_proba(Xe)[:, 1]
y_edge = edge["default"].values
approve_edge = (p_edge < 1/9).astype(int)

print("Default rate in edge group:", y_edge.mean().round(3))
print("Model cost: ", total_cost(y_edge, approve_edge))
print("Approve all:", total_cost(y_edge, np.ones(len(y_edge))))
print("Reject all: ", total_cost(y_edge, np.zeros(len(y_edge))))

Default rate in edge group: 0.134
Model cost:  1484
Approve all: 1816
Reject all:  1469


In [67]:
ok = p_rej < 1/9    # the rejected people the model would approve
rows = []
for factor in [1, 1.5, 2, 3]:
    p_true = np.clip(p_rej * factor, 0, 1)
    policy = 8 * p_true[ok].sum() + (1 - p_true[~ok]).sum()
    rows.append({"real risk is x times model's": factor,
                 "our policy": round(policy),
                 "approve all": round(8 * p_true.sum()),
                 "reject all": round((1 - p_true).sum())})

print(pd.DataFrame(rows).to_string(index=False))

 real risk is x times model's  our policy  approve all  reject all
                          1.0        9068        29721        9285
                          1.5        7637        44365        7454
                          2.0        6421        57300        5837
                          3.0        4906        75815        3523


In [68]:
small_loan = unlabeled["loan_amt"] <= labeled["loan_amt"].median()

pilot = unlabeled[unlabeled["seen_range"] & (unlabeled["p_default"] < 0.08) & small_loan]
print("Rejected people eligible for the pilot:", len(pilot))

pilot = pilot.sample(min(300, len(pilot)), random_state=42)
print("Pilot size:", len(pilot))
print("Expected defaults in pilot:", round(pilot["p_default"].sum(), 1))
print("Average default chance in pilot:", round(pilot["p_default"].mean(), 3))

Rejected people eligible for the pilot: 123
Pilot size: 123
Expected defaults in pilot: 8.2
Average default chance in pilot: 0.067
